### Notebook scope

### Purpose
Compare paired INT and NOCOUPL hindcasts.

### Scientific question
Does interactive O3 systematically alter O3 loss, vortex winds, EP100, or persistence?

### Method
Pair same year/init small-temperature INT and NOCOUPL cases only. Compute INT minus NOCOUPL ensemble-mean differences with bootstrap CI where practical.

### Expected output
Per-pair figures, overview figure, and metrics CSV.

### Interpretation
Sustained negative Delta O3 and positive/negative wind differences indicate feedback on ozone-loss persistence and vortex pathway.

### Caveat
0003 is not forced into paired comparison if no NOCOUPL exists.


### Setup imports and shared paths

### Purpose
Load the shared Extention_analysis utility module and create output directories.

### Scientific question
Can every notebook start from a clean kernel and still find the same data roots and output roots?

### Method
Use DATA_ROOT, HINDCAST_ROOT, BWCN_ROOT, B2000_ROOT, WORK_ROOT, FIG_DIR, TAB_DIR, CACHE_DIR, and LOG_DIR from hindcast_extension_utils.py. No diagnostics are calculated in this cell.

### Expected output
Printed path check only. No figure is generated by this setup cell.

### Interpretation
Successful execution means the notebook can access the common utilities and write outputs in the agreed directory tree.

### Caveat
This setup does not prove that every downstream data product exists; missing products are logged later.


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter

import hindcast_extension_utils as hx
from hindcast_extension_utils import *
_assign_member_short = hx._assign_member_short

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)

### Compute paired INT minus NOCOUPL evolution

### Purpose
Create paired feedback evolution plots for all available small-perturbation pairs.

### Scientific question
Does interactive O3 maintain lower O3 or a more persistent vortex relative to NOCOUPL?

### Method
Variables are O3, U60N10, U60N50, and EP100 W1+W2. Differences are INT minus NOCOUPL after aligning members by id when possible; otherwise ensemble means are compared.

### Expected output
outputs/figures/07_INT_minus_NOCOUPL_evolution_<year>_<init>.png/pdf and outputs/tables/07_INT_minus_NOCOUPL_evolution_metrics.csv.

### Interpretation
Persistent nonzero differences after initialization suggest interactive-O3 feedback, not just initial condition noise.

### Caveat
Member pairing may be imperfect; unpaired comparisons use ensemble mean differences and bootstrap CIs are approximate.


In [ ]:
inv = discover_hindcast_cases()
pairs = paired_int_nocoupl_cases(inv)
rows = []

def member_mean_series(da):
    return da.mean("member", skipna=True) if da is not None and "member" in da.dims else None

def align_diff(int_da, nc_da):
    if int_da is None or nc_da is None:
        return None
    int_da = _assign_member_short(int_da).swap_dims({"member":"member_short"}) if "member_short" in int_da.coords else int_da
    nc_da = _assign_member_short(nc_da).swap_dims({"member":"member_short"}) if "member_short" in nc_da.coords else nc_da
    if "member_short" in int_da.dims and "member_short" in nc_da.dims:
        common = sorted(set(int_da["member_short"].values) & set(nc_da["member_short"].values))
        if len(common) >= 3:
            n = min(int_da.sizes["lead_time"], nc_da.sizes["lead_time"])
            return int_da.sel(member_short=common).isel(lead_time=slice(0,n)) - nc_da.sel(member_short=common).isel(lead_time=slice(0,n))
    n = min(int_da.sizes["lead_time"], nc_da.sizes["lead_time"])
    return int_da.isel(lead_time=slice(0,n)).mean("member") - nc_da.isel(lead_time=slice(0,n)).mean("member")

for _, pair in pairs.iterrows():
    icase, ncase = pair["INT_case"], pair["NOCOUPL_case"]
    year, init = pair["year"], pair["init_month"]
    o3_i, date_i = load_hindcast_o3(icase); o3_n, date_n = load_hindcast_o3(ncase)
    u10_i, _ = compute_u60(icase, 10); u10_n, _ = compute_u60(ncase, 10)
    u50_i, _ = compute_u60(icase, 50); u50_n, _ = compute_u60(ncase, 50)
    ep_i = compute_all_ep100(icase, source_windows_for_case(icase)["primary"])
    ep_n = compute_all_ep100(ncase, source_windows_for_case(ncase)["primary"])
    fig, axes = plt.subplots(4,1,figsize=(10.5,12),sharex=False,constrained_layout=True)
    products = [("O3", align_diff(o3_i,o3_n), "DU"), ("U60N10", align_diff(u10_i,u10_n), "m s-1"), ("U60N50", align_diff(u50_i,u50_n), "m s-1")]
    for ax, (var, diff, unit) in zip(axes[:3], products):
        if diff is None:
            ax.text(0.5,0.5,f"missing {var}",transform=ax.transAxes,ha="center")
            continue
        vals = diff.values if "member_short" in diff.dims else diff.values[None,:]
        mean = np.nanmean(vals, axis=0); std = np.nanstd(vals, axis=0)
        x = np.arange(len(mean))
        ax.plot(x, mean, color="tab:red", lw=2); ax.fill_between(x, mean-std, mean+std, color="tab:red", alpha=0.18); ax.axhline(0,color="0.3",lw=0.8)
        ax.set_ylabel(f"Delta {var} ({unit})")
        rows.append({"year":year,"init_month":init,"INT_case":icase,"NOCOUPL_case":ncase,"variable":var,"mean_delta":float(np.nanmean(mean))})
    if not ep_i.empty and not ep_n.empty:
        common = sorted(set(ep_i["member"]) & set(ep_n["member"]))
        if common:
            delta = ep_i.set_index("member").loc[common,"EP100_wave1_plus_wave2"] - ep_n.set_index("member").loc[common,"EP100_wave1_plus_wave2"]
        else:
            delta = pd.Series([ep_i["EP100_wave1_plus_wave2"].mean() - ep_n["EP100_wave1_plus_wave2"].mean()])
        axes[3].bar(np.arange(len(delta)), delta.values, color="tab:purple"); axes[3].axhline(0,color="0.3",lw=0.8); axes[3].set_ylabel("Delta EP100 W1+W2")
        rows.append({"year":year,"init_month":init,"INT_case":icase,"NOCOUPL_case":ncase,"variable":"EP100_W1W2","mean_delta":float(delta.mean())})
    fig.suptitle(f"INT minus NOCOUPL evolution: {year}-{init}")
    case_csv = TAB_DIR / f"07_INT_minus_NOCOUPL_evolution_{year}_{init}.csv"
    pd.DataFrame([r for r in rows if r["year"]==year and r["init_month"]==init]).to_csv(case_csv,index=False)
    savefig(fig, f"07_INT_minus_NOCOUPL_evolution_{year}_{init}", notebook="07_INT_vs_NOCOUPL_feedback.ipynb", scientific_question="Does INT differ systematically from NOCOUPL for this paired case?", variables_windows="INT-NOCOUPL; O3, U60N10, U60N50 time series; EP100 W1+W2 source window", interpretation="Persistent nonzero Delta values indicate interactive-O3 feedback on ozone/vortex/wave pathway.", caveat="Member pairing uses common member ids when available; otherwise ensemble means are compared.", csv_table=case_csv)
    plt.close(fig)
metrics = pd.DataFrame(rows)
metrics_csv = TAB_DIR / "07_INT_minus_NOCOUPL_evolution_metrics.csv"
metrics.to_csv(metrics_csv,index=False)
fig, ax = plt.subplots(figsize=(11,5),constrained_layout=True)
if metrics.empty:
    ax.axis("off"); ax.text(0.5,0.5,"No paired INT/NOCOUPL cases",ha="center")
else:
    piv = metrics.pivot_table(index=["year","init_month"], columns="variable", values="mean_delta")
    im = ax.imshow(piv.values, cmap="RdBu_r", aspect="auto")
    ax.set_xticks(range(piv.shape[1]), piv.columns, rotation=30, ha="right"); ax.set_yticks(range(piv.shape[0]), [f"{a}-{b}" for a,b in piv.index])
    fig.colorbar(im, ax=ax, label="INT - NOCOUPL mean delta")
    ax.set_title("INT minus NOCOUPL overview")
savefig(fig, "07_INT_minus_NOCOUPL_evolution_overview", notebook="07_INT_vs_NOCOUPL_feedback.ipynb", scientific_question="Are INT-NOCOUPL feedback differences systematic across pairs?", variables_windows="Paired small-perturbation cases; O3/U/EP100 metrics", interpretation="Same-sign deltas across pairs suggest systematic interactive-O3 feedback.", caveat="0003 is excluded unless a true paired NOCOUPL case exists.", csv_table=metrics_csv)
plt.show()
write_figure_guide()